# Steam Oyun Benzerlik Ağı – Analiz, Bölüm 2: Bağlantı Tahmini
Beril Özdizdar · Sosyal Ağ Analizi dönem ödevi

**Araştırma sorusu:** Bağımsız bir stüdyo için doymamış (az rekabetli ama ilgi gören) etiket kombinasyonları hangileri?

**Fikir:** Etiketler arasında bir ağ kurarız: iki etiket, en az 3 oyunda birlikte geçiyorsa aralarında kenar vardır (yani "bu kombinasyon piyasada var"). Henüz kenarı olmayan ama ağdaki konumu gereği **olası** görünen çiftleri **bağlantı tahmini** ile buluruz:
- **Adamic-Adar:** İki etiketin ortak komşuları varsa, o komşu az bağlantılıysa (özel) daha çok sayılır.
- **Jaccard:** İki etiketin komşu kümelerinin ne kadar örtüştüğü.

**Küçük örnek (sunumda anlat):** X etiketinin komşuları {Z1, Z2, P}, Y etiketinin komşuları {Z1, Z2, Q, R} olsun. Ortak komşu 2 tane (Z1, Z2), birleşim 5 etiket → Jaccard = 2/5 = 0.4.
Z1'in 2, Z2'nin 5 komşusu varsa Adamic-Adar = 1/ln(2) + 1/ln(5) = 1.44 + 0.62 = 2.06. X ile Y henüz birlikte kullanılmıyor ama ortak komşuları çok, yani "olmayan ama olası" bir kombinasyon.

**Doğrulama:** Oyunları kimlik numarasına (appid) göre eski (%70) ve yeni (%30) diye ayırırız (yeni oyunların numarası daha büyüktür). Eski oyunlardan ağı kurup tahmin yaparız, sonra yeni oyunlarda bu çiftlerin gerçekten ortaya çıkıp çıkmadığına bakarız.

## 0. Kurulum ve veri

In [ ]:
import os, glob, json, subprocess, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from google.colab import files
warnings.filterwarnings("ignore")

if not glob.glob("/content/**/processed/edges_game_tag.csv", recursive=True):
    if not glob.glob("/content/steam-network*.zip"):
        print("Açılan pencereden steam-network.zip dosyasını seç:")
        files.upload()
    for z in glob.glob("/content/steam-network*.zip"):
        subprocess.run(["unzip", "-q", "-o", z, "-d", "/content"])

hit = glob.glob("/content/**/processed/edges_game_tag.csv", recursive=True)
assert hit, "Veri bulunamadı: zip'in içinde data/processed klasörü olduğundan emin ol."
BASE = os.path.dirname(os.path.dirname(hit[0]))
OUT = "/content/sonuclar2"
os.makedirs(OUT, exist_ok=True)
print("Veri bulundu:", BASE)

In [ ]:
gt = pd.read_csv(f"{BASE}/processed/edges_game_tag.csv", encoding="utf-8-sig")

meta = {}
for line in open(f"{BASE}/raw/appdetails.jsonl", encoding="utf-8"):
    if line.strip():
        d = json.loads(line)
        meta[d["appid"]] = d.get("positive", 0) + d.get("negative", 0)

app_ids = np.array(sorted(gt["appid"].unique()))
tags = sorted(gt["etiket"].unique())
aix = {a: i for i, a in enumerate(app_ids)}
tix = {t: i for i, t in enumerate(tags)}

# oyun x etiket ikili matrisi
X = np.zeros((len(app_ids), len(tags)))
for a, t in zip(gt["appid"], gt["etiket"]):
    X[aix[a], tix[t]] = 1

nT = len(tags)
iu = np.triu_indices(nT, 1)
print(f"{len(app_ids)} oyun, {nT} etiket")

## 1. Bağlantı tahmini fonksiyonları

In [ ]:
def birlikte_sayim(Xs):
    M = Xs.T @ Xs          # M[i,j] = i ve j etiketlerinin birlikte geçtiği oyun sayısı
    np.fill_diagonal(M, 0)
    return M

def skorlar(A):
    A = A.astype(float)
    deg = A.sum(1)
    cn = A @ A                                           # ortak komşu sayısı
    w = np.where(deg > 1, 1.0 / np.log(np.maximum(deg, 2)), 0.0)
    aa = (A * w) @ A                                     # Adamic-Adar
    union = deg[:, None] + deg[None, :] - cn
    jac = np.divide(cn, union, out=np.zeros_like(cn), where=union > 0)   # Jaccard
    pa = np.outer(deg, deg)                              # tercihli bağlanma (karşılaştırma için)
    return {"Adamic-Adar": aa, "Jaccard": jac, "Ortak komşu": cn, "Tercihli bağlanma": pa}

def precision_at_k(y, s, k):
    top = np.argsort(-s, kind="stable")[:k]
    return y[top].mean()

## 2. Geçmiş verilerle doğrulama (eski %70 → yeni %30)

In [ ]:
CUT_Q = 0.70                                  # eski/yeni ayrımı
cut_id = app_ids[int(len(app_ids) * CUT_Q)]
old = app_ids <= cut_id
print(f"Eski oyun: {old.sum()} | Yeni oyun: {(~old).sum()} | kesim appid: {cut_id}")

M_old = birlikte_sayim(X[old])
M_all = birlikte_sayim(X)

def dogrula(m):
    A_old = M_old >= m
    A_all = M_all >= m
    mask = ~A_old[iu]                       # eski ağda kenarı olmayan çiftler = adaylar
    y = A_all[iu][mask].astype(int)         # sonradan kenar olanlar = doğru tahmin
    if y.sum() == 0 or y.sum() == len(y):
        return None, y
    sc = skorlar(A_old)
    rows = []
    for ad, S in sc.items():
        s = S[iu][mask]
        rows.append({"eşik m": m, "yöntem": ad,
                     "AUC": round(roc_auc_score(y, s), 3),
                     "AP": round(average_precision_score(y, s), 3),
                     "P@50": round(precision_at_k(y, s, 50), 3),
                     "P@100": round(precision_at_k(y, s, 100), 3)})
    return pd.DataFrame(rows), y

M_ESIK = 3     # "kombinasyon var" demek için gereken en az oyun sayısı
tablo, y = dogrula(M_ESIK)
print(f"Aday çift: {len(y)} | sonradan ortaya çıkan: {y.sum()} | rastgele tahminle isabet oranı: {y.mean():.3f}")
tablo.to_csv(f"{OUT}/dogrulama_m{M_ESIK}.csv", index=False, encoding="utf-8-sig")
tablo

In [ ]:
# Dayanıklılık: farklı eşiklerle de aynı sonuç çıkıyor mu?
parcalar = []
for m in [2, 3, 5]:
    t, _ = dogrula(m)
    if t is not None:
        parcalar.append(t)
duyarlilik = pd.concat(parcalar, ignore_index=True)
duyarlilik.to_csv(f"{OUT}/dogrulama_duyarlilik.csv", index=False, encoding="utf-8-sig")
duyarlilik.pivot(index="yöntem", columns="eşik m", values="AUC")

**Yorum (kendi cümlelerinle yaz):**
- AUC 0.5 rastgele tahmin demek. Adamic-Adar ve Jaccard kaç çıktı? Rastgeleden ne kadar iyi? …
- P@100: tahmin ettiğimiz en olası 100 çiftin yüzde kaçı gerçekten sonradan ortaya çıktı? Rastgele isabet oranıyla (yukarıdaki satır) kıyasla. …
- Hangi yöntem daha iyi, neden olabilir? …
- Sınırlama: appid gerçek çıkış tarihi değil, sadece yaklaşık bir zaman sırası. …

## 3. Araştırma sorusunun cevabı: olmayan ama olası kombinasyonlar

Tüm oyunlarla ağı kuruyoruz (3'ten az oyunda birlikte geçen çiftler "yok" sayılır) ve kenarı olmayan çiftleri skorluyoruz. Üç ölçü birlikte okunur:
- **Olasılık:** Adamic-Adar skoru yüksek (ağdaki konumu gereği olası)
- **Rekabet:** Kombinasyonu içeren oyun sayısı az (0, 1 veya 2)
- **Talep:** İki etiketin geçtiği oyunların ortalama değerlendirme sayısı (log10) yüksek, yani oyuncu ilgisi var

In [ ]:
A = M_all >= M_ESIK
sc = skorlar(A)
mask = (~A)[iu]

# talep: etiketi taşıyan oyunların ortalama log10(değerlendirme + 1) değeri
log_rev = np.array([np.log10(1 + meta.get(a, 0)) for a in app_ids])
talep_tag = (X.T @ log_rev) / X.sum(0)

aday = pd.DataFrame({
    "etiket_a": [tags[i] for i in iu[0][mask]],
    "etiket_b": [tags[j] for j in iu[1][mask]],
    "oyun_sayisi": M_all[iu][mask].astype(int),          # rekabet
    "adamic_adar": sc["Adamic-Adar"][iu][mask].round(3),  # olasılık
    "jaccard": sc["Jaccard"][iu][mask].round(3),
    "talep": ((talep_tag[iu[0]] + talep_tag[iu[1]]) / 2)[mask].round(3),
})
aday.to_csv(f"{OUT}/tum_adaylar.csv", index=False, encoding="utf-8-sig")

esik_aa = aday["adamic_adar"].quantile(0.95)
esik_talep = aday["talep"].median()
oneri = (aday[(aday["adamic_adar"] >= esik_aa) & (aday["talep"] >= esik_talep)]
         .sort_values("adamic_adar", ascending=False).head(25).reset_index(drop=True))
oneri.to_csv(f"{OUT}/doymamis_kombinasyonlar_top25.csv", index=False, encoding="utf-8-sig")
print(f"Aday çift: {len(aday)} | olasılık eşiği (üst %5): {esik_aa:.2f} | talep eşiği (medyan): {esik_talep:.2f}")
oneri

**Yorum:** Tablodaki ilk 5 kombinasyonu seç ve her biri için bir cümle yaz: bu iki etiket neden birlikte "olası" (ortak komşuları ne?), neden henüz az oyunda var (rekabet düşük), neden talep var? Bağımsız bir stüdyoya ne önerirsin? …

**Nedensellik uyarısı:** Bu ağ verisi "bu kombinasyon başarılı olur" demez, yalnızca benzer etiketlerin çevresinde henüz boş kalmış yerleri gösterir.

### Karşılaştırma için: en doymuş kombinasyonlar

In [ ]:
doymus = pd.DataFrame({
    "etiket_a": [tags[i] for i in iu[0]],
    "etiket_b": [tags[j] for j in iu[1]],
    "oyun_sayisi": M_all[iu].astype(int),
}).sort_values("oyun_sayisi", ascending=False).head(15).reset_index(drop=True)
doymus.to_csv(f"{OUT}/en_doymus_kombinasyonlar_top15.csv", index=False, encoding="utf-8-sig")
doymus

## 4. Sonuçları indir

In [ ]:
import shutil
shutil.make_archive("/content/sonuclar2", "zip", OUT)
files.download("/content/sonuclar2.zip")